This notebook is used to generate the subchannel images that we need for hand labelling so we can upload them to Roboflow and annotate them.

In [ ]:
import tifffile
import glob 
import os
import cv2
import dataset_utils


This extracts the necessary spectral data to do manual annotation

In [ ]:
def extract_important_features(filename, dest_folder):
    band_list = [[5,3,2], [10,7,3], [10], [11], [12]]
    rgb = dataset_utils.load_tif_image(filename, band_list[0])
    fire = dataset_utils.load_tif_image(filename, band_list[1])
    swir2 = cv2.applyColorMap(dataset_utils.load_tif_image(filename, band_list[2]), cv2.COLORMAP_VIRIDIS) 
    ir = cv2.applyColorMap(dataset_utils.load_tif_image(filename, band_list[3]), cv2.COLORMAP_VIRIDIS) 
    thermal = cv2.applyColorMap(dataset_utils.load_tif_image(filename, band_list[4]), cv2.COLORMAP_VIRIDIS) 

    basename = os.path.basename(filename)[:-4] # get rid of .tif

    if not os.path.exists(f'{dest_folder}/{basename}/'):
        os.mkdir(f'{dest_folder}/{basename}')

    tifffile.imwrite(f'{dest_folder}/{basename}/RGB_{basename}.tif', rgb)
    tifffile.imwrite(f'{dest_folder}/{basename}/FIRE_EMPHASIS_{basename}.tif', fire)
    tifffile.imwrite(f'{dest_folder}/{basename}/SWIR2_{basename}.tif', swir2)
    tifffile.imwrite(f'{dest_folder}/{basename}/IR_{basename}.tif', ir)
    tifffile.imwrite(f'{dest_folder}/{basename}/THERMAL_{basename}.tif', thermal)


def generate_labelling_data(base_folder, dest_folder):
    complete_images = glob.glob(f'{base_folder}/*.tif')
    for img in complete_images:
        extract_important_features(img, dest_folder)

    


        

In [ ]:
generate_labelling_data('./data/ams_data/processed_images/complete_images', './data/ams_data/processed_images/labelling_data')

In [ ]:
def convert_to_png(folder):
    tif_files = glob.glob(f'{folder}/*/*.tif')
    print(len(tif_files))
    for file in tif_files:
        img = tifffile.imread(file)
        if img is None:
            raise Exception(f'Issue with reading file {file}')
        img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
        cv2.imwrite(f'{file[:-4]}.png', img)

In [ ]:
convert_to_png('./data/ams_data/processed_images/labelling_data')

In [ ]:
folders = glob.glob('./data/ams_data/processed_images/labelling_data/*/')
folders.sort()
print(len(folders))
print(folders[len(folders)//2])